## UrbanRide_04 — Silver Trips
**Method:** Batch transformation — Bronze → Silver  
**Input:** `urbanride.bronze.trips`  
**Output:** `urbanride.silver.trips`  
**Schedule:** Daily · runs after UrbanRide_02 (Bronze Events ingest)

**Load strategy:**
- First run → full load
- Daily runs → incremental append
- Watermark: `_ingested_at` (Bronze) vs `_processed_at` (Silver)
- Target grain: one clean row per `trip_id`

**Transformations:**
- Deduplication on `trip_id`
- Cast `pickup_time`, `drop_time` STRING → TIMESTAMP
- Calculate `trip_duration_minutes`
- Flag negative `distance_km` → `is_distance_invalid`
- Flag ghost trips — `distance_km = 0` + `trip_status = completed`
- Validate `trip_status` — only `completed` / `cancelled`
- Partition by `trip_status`

In [0]:
from pyspark.sql.functions import (
    col, when, lit, current_timestamp,
    to_timestamp, unix_timestamp, round,
    count, sum as spark_sum
)

CATALOG = 'urbanride'
BRONZE  = f'{CATALOG}.bronze'
SILVER  = f'{CATALOG}.silver'

spark.sql(f'USE CATALOG {CATALOG}')

VALID_STATUSES = ['completed', 'cancelled']
VALID_WEATHER  = ['Clear', 'Rain', 'Fog']
VALID_VEHICLES = ['Auto', 'Sedan', 'SUV']
VALID_DAYTYPES = ['weekday', 'weekend', 'holiday']

print(f'Catalog : {CATALOG}')
print(f'Source  : {BRONZE}.trips')
print(f'Target  : {SILVER}.trips')
print('Config ready.')


Catalog : urbanride
Source  : urbanride.bronze.trips
Target  : urbanride.silver.trips
Config ready.


In [0]:
print('Checking load type...')

table_exists = spark.catalog.tableExists(f'{SILVER}.trips')

if not table_exists:
    LOAD_TYPE = 'full'
    last_run  = None
    print('  Silver table does not exist — FULL LOAD')
else:
    LOAD_TYPE = 'incremental'
    last_run  = spark.sql(
        f'SELECT MAX(_processed_at) FROM {SILVER}.trips'
    ).collect()[0][0]
    print(f'  Silver table exists — INCREMENTAL LOAD')
    print(f'  Last processed at : {last_run}')

print(f'  Load type : {LOAD_TYPE}')


Checking load type...
  Silver table does not exist — FULL LOAD
  Load type : full


In [0]:
print('Reading bronze.trips...')

df_bronze_full = spark.table(f'{BRONZE}.trips')

if LOAD_TYPE == 'full':
    df_bronze = df_bronze_full
    print(f'  Full load — reading all rows')
else:
    df_bronze = df_bronze_full.filter(col('_ingested_at') > last_run)
    print(f'  Incremental — reading rows ingested after {last_run}')

input_count = df_bronze.count()
print(f'  Rows to process : {input_count:,}')

if input_count == 0:
    print('  No new rows — Silver already up to date.')
    dbutils.notebook.exit('No new rows — skipping')

# Profile before cleaning
print()
print('Bronze profile (before cleaning):')
print(f'  Negative distances : {df_bronze.filter(col("distance_km") < 0).count():,}')
print(f'  Ghost trips        : {df_bronze.filter((col("distance_km") == 0) & (col("trip_status") == "completed")).count():,}')
print(f'  Invalid status     : {df_bronze.filter(~col("trip_status").isin(VALID_STATUSES)).count():,}')
print(f'  Duplicate trip_ids : {input_count - df_bronze.dropDuplicates(["trip_id"]).count():,}')
print()
print('Trip status distribution:')
df_bronze.groupBy('trip_status').count().show()
print('Weather distribution:')
df_bronze.groupBy('weather').count().orderBy('count', ascending=False).show()
print('Day type distribution:')
df_bronze.groupBy('day_type').count().orderBy('count', ascending=False).show()


Reading bronze.trips...
  Full load — reading all rows
  Rows to process : 21,359,745

Bronze profile (before cleaning):
  Negative distances : 64,336
  Ghost trips        : 39,067
  Invalid status     : 0
  Duplicate trip_ids : 0

Trip status distribution:
+-----------+--------+
|trip_status|   count|
+-----------+--------+
|  cancelled| 1817529|
|  completed|19542216|
+-----------+--------+

Weather distribution:
+-------+--------+
|weather|   count|
+-------+--------+
|  Clear|13858219|
|   Rain| 5364869|
|    Fog| 2136657|
+-------+--------+

Day type distribution:
+--------+--------+
|day_type|   count|
+--------+--------+
| weekday|14623416|
| weekend| 4493356|
| holiday| 2242973|
+--------+--------+



In [0]:
print('Deduplicating on trip_id...')

df_deduped = df_bronze.dropDuplicates(['trip_id'])

removed = input_count - df_deduped.count()
print(f'  Rows before : {input_count:,}')
print(f'  Rows after  : {df_deduped.count():,}')
print(f'  Removed     : {removed:,} duplicate rows')


Deduplicating on trip_id...
  Rows before : 21,359,745
  Rows after  : 21,359,745
  Removed     : 0 duplicate rows


In [0]:
# pickup_time and drop_time are STRING in Bronze — cast to TIMESTAMP
# Bronze stores raw string values — Silver enforces correct types
print('Casting timestamps...')

df_typed = df_deduped \
    .withColumn('pickup_time', to_timestamp(col('pickup_time'), 'yyyy-MM-dd HH:mm:ss')) \
    .withColumn('drop_time',   to_timestamp(col('drop_time'),   'yyyy-MM-dd HH:mm:ss'))

# Check for nulls introduced by bad timestamp format
bad_pickup = df_typed.filter(col('pickup_time').isNull()).count()
bad_drop   = df_typed.filter(col('drop_time').isNull()).count()

print(f'  NULL pickup_time after cast : {bad_pickup:,}')
print(f'  NULL drop_time after cast   : {bad_drop:,}')

if bad_pickup > 0 or bad_drop > 0:
    print('  WARNING — some timestamps could not be parsed')
    df_typed.filter(col('pickup_time').isNull()) \
        .select('trip_id','pickup_time','drop_time').show(5)


Casting timestamps...
  NULL pickup_time after cast : 0
  NULL drop_time after cast   : 0


In [0]:
# Calculate trip_duration_minutes from pickup and drop timestamps
# This is a derived feature — useful for demand forecasting Gold table
print('Calculating trip duration...')

df_duration = df_typed.withColumn(
    'trip_duration_minutes',
    round(
        (unix_timestamp(col('drop_time')) - unix_timestamp(col('pickup_time'))) / 60,
        2
    )
)

# Flag negative durations — drop_time before pickup_time
df_duration = df_duration.withColumn(
    'is_duration_invalid',
    col('trip_duration_minutes') < 0
)

print('Trip duration stats:')
df_duration.select('trip_duration_minutes') \
    .summary('min','max','mean','stddev') \
    .show()

invalid_dur = df_duration.filter(col('is_duration_invalid') == True).count()
print(f'  Negative duration trips : {invalid_dur:,}')


Calculating trip duration...
Trip duration stats:
+-------+---------------------+
|summary|trip_duration_minutes|
+-------+---------------------+
|    min|                 10.0|
|    max|                 40.0|
|   mean|    24.99788845793805|
| stddev|    8.944140074609043|
+-------+---------------------+

  Negative duration trips : 0


In [0]:
# Flag data quality issues — don't drop, mark them
# Silver preserves all rows — Gold decides what to use
print('Flagging distance issues and ghost trips...')

df_flags = df_duration \
    .withColumn('is_distance_invalid',
        col('distance_km') < 0
    ) \
    .withColumn('is_ghost_trip',
        # Ghost trip — distance = 0 but status = completed
        # is_ghost_trip already exists from Bronze generator
        # recompute here as single source of truth in Silver
        when(
            (col('distance_km') == 0) & (col('trip_status') == 'completed'), True
        ).otherwise(False)
    )

print(f'  Negative distance trips : {df_flags.filter(col("is_distance_invalid") == True).count():,}')
print(f'  Ghost trips             : {df_flags.filter(col("is_ghost_trip") == True).count():,}')
print(f'  Fare inflated trips     : {df_flags.filter(col("is_fare_inflated") == True).count():,}')


Flagging distance issues and ghost trips...
  Negative distance trips : 64,336
  Ghost trips             : 39,067
  Fare inflated trips     : 21,201


In [0]:
# Flag any trip_status values outside expected set
# completed and cancelled are the only valid values
print('Validating trip status...')

df_validated = df_flags.withColumn(
    'is_status_invalid',
    ~col('trip_status').isin(VALID_STATUSES)
)

invalid_status = df_validated.filter(col('is_status_invalid') == True).count()
print(f'  Invalid status rows : {invalid_status:,}')

if invalid_status > 0:
    df_validated.filter(col('is_status_invalid') == True) \
        .groupBy('trip_status').count().show()


Validating trip status...
  Invalid status rows : 0


In [0]:
df_silver = df_validated \
    .withColumn('_processed_at', current_timestamp()) \
    .withColumn('_source', lit(f'{BRONZE}.trips'))

print(f'Final row count   : {df_silver.count():,}')
print(f'Final column count: {len(df_silver.columns)}')
print()
print('Final schema:')
df_silver.printSchema()


Final row count   : 21,359,745
Final column count: 22

Final schema:
root
 |-- trip_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- pickup_time: timestamp (nullable = true)
 |-- drop_time: timestamp (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- surge_multiplier: double (nullable = true)
 |-- trip_status: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- weather: string (nullable = true)
 |-- is_ghost_trip: boolean (nullable = false)
 |-- is_fare_inflated: boolean (nullable = true)
 |-- day_type: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- is_duration_invalid: boolean (nullable = true)
 |-- is_distance_invalid: boolean (nullable = true)
 |-- is_status_invalid: boolean (nullable = true)
 |-- _processed_at: timesta

In [0]:
import time
print(f'Writing silver.trips — mode: {LOAD_TYPE}...')
t0 = time.time()

if LOAD_TYPE == 'full':
    df_silver.write \
        .format('delta') \
        .mode('overwrite') \
        .partitionBy('trip_status') \
        .option('overwriteSchema', 'true') \
        .saveAsTable(f'{SILVER}.trips')
    print(f'  Full load written : {df_silver.count():,} rows')
    print(f'  Write time : {time.time()-t0}s')
    print('Running OPTIMIZE...')
    spark.sql(f'OPTIMIZE {SILVER}.trips')
    print('  OPTIMIZE complete')
else:
    new_count = df_silver.count()
    if new_count > 0:
        df_silver.write \
            .format('delta') \
            .mode('append') \
            .saveAsTable(f'{SILVER}.trips')
        print(f'  Incremental rows appended : {new_count:,}')
        print(f'  Write time : {time.time()-t0}s')
        print('Running OPTIMIZE...')
        spark.sql(f'OPTIMIZE {SILVER}.trips')
        print('  OPTIMIZE complete')
    else:
        print('  No new rows — skipping write and OPTIMIZE')


Writing silver.trips — mode: full...
  Full load written : 21,359,745 rows
  Write time : 37.558573722839355s
Running OPTIMIZE...
  OPTIMIZE complete


In [0]:
print('=== SILVER TRIPS VERIFICATION ===')
print()

df_verify = spark.table(f'{SILVER}.trips')
total = df_verify.count()

print(f'  Total rows : {total:,}')
print(f'  Load type  : {LOAD_TYPE}')
print()

# Data quality checks
print('Data quality checks:')
checks = [
    ('Null trip_id',           df_verify.filter(col('trip_id').isNull()).count()),
    ('Null pickup_time',       df_verify.filter(col('pickup_time').isNull()).count()),
    ('Null drop_time',         df_verify.filter(col('drop_time').isNull()).count()),
    ('Negative distance',      df_verify.filter(col('is_distance_invalid') == True).count()),
    ('Ghost trips',            df_verify.filter(col('is_ghost_trip') == True).count()),
    ('Fare inflated',          df_verify.filter(col('is_fare_inflated') == True).count()),
    ('Invalid status',         df_verify.filter(col('is_status_invalid') == True).count()),
    ('Negative duration',      df_verify.filter(col('is_duration_invalid') == True).count()),
]

for check, result in checks:
    # These are expected fraud/quality signals — WARN not FAIL
    expected = ['Negative distance','Ghost trips','Fare inflated']
    if check in expected:
        status = 'INFO' if result > 0 else 'PASS'
    else:
        status = 'WARN' if result > 0 else 'PASS'
    print(f'  {status}  {check:<30} : {result:,}')

print()
print('Trip status distribution:')
df_verify.groupBy('trip_status').count().orderBy('count', ascending=False).show()

print('Weather distribution:')
df_verify.groupBy('weather').count().orderBy('count', ascending=False).show()

print('Day type distribution:')
df_verify.groupBy('day_type').count().orderBy('count', ascending=False).show()

print('Duration stats (completed trips only):')
df_verify.filter(col('trip_status') == 'completed') \
    .select('trip_duration_minutes') \
    .summary('min','max','mean') \
    .show()

print('Sample rows:')
df_verify.select(
    'trip_id','city','trip_status','distance_km',
    'fare_amount','weather','day_type',
    'trip_duration_minutes','is_distance_invalid',
    'is_ghost_trip','is_fare_inflated'
).limit(5).show(truncate=False)

print()
print('Silver trips ready. Next: UrbanRide_05 — Silver Payments')


=== SILVER TRIPS VERIFICATION ===

  Total rows : 21,359,745
  Load type  : full

Data quality checks:
  PASS  Null trip_id                   : 0
  PASS  Null pickup_time               : 0
  PASS  Null drop_time                 : 0
  INFO  Negative distance              : 64,336
  INFO  Ghost trips                    : 39,067
  INFO  Fare inflated                  : 21,201
  PASS  Invalid status                 : 0
  PASS  Negative duration              : 0

Trip status distribution:
+-----------+--------+
|trip_status|   count|
+-----------+--------+
|  completed|19542216|
|  cancelled| 1817529|
+-----------+--------+

Weather distribution:
+-------+--------+
|weather|   count|
+-------+--------+
|  Clear|13858219|
|   Rain| 5364869|
|    Fog| 2136657|
+-------+--------+

Day type distribution:
+--------+--------+
|day_type|   count|
+--------+--------+
| weekday|14623416|
| weekend| 4493356|
| holiday| 2242973|
+--------+--------+

Duration stats (completed trips only):
+-------+----

In [0]:
display(spark.sql("DESCRIBE HISTORY urbanride.silver.trips"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-03-09T16:38:56.000Z,8815326091183894,bvishaladf@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [""trip_status""], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(782468310062608),c4bc2a14-8b74-44ce-935b-db777ac4bcec,0309-161839-7p4nh3de-v2n,null,WriteSerializable,false,"Map(numFiles -> 7, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 21359745, numOutputBytes -> 1117430918)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
